# Temporal Analysis of Ukraine Conflict Data (2022-2025)

## Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import sys

import matplotlib.pyplot as plt
import networkx as nx
from node2vec import Node2Vec
from sklearn.decomposition import PCA

sys.path.append("../")

from core.functions import (
    normalize_actor,
    build_actor_interaction_graph,
    add_pairwise_embedding_features,
)

## Set path and read in data

In [ ]:
data_path = "../data/raw/"

In [ ]:
# read in the data and set the index as 'event_id_cnty'
df = pd.read_csv(os.path.join(data_path, "ACLED Data_2026-01-02.csv"))

In [ ]:
df = df.set_index("event_id_cnty")

## Visualizations of temporal trends and patterns

In [ ]:
# Convert event_date to datetime
df["event_date"] = pd.to_datetime(df["event_date"])

# Basic temporal statistics
print("Date Range:")
print(f"Start: {df['event_date'].min()}")
print(f"End: {df['event_date'].max()}")
print(f"Total Days: {(df['event_date'].max() - df['event_date'].min()).days}")
print(f"Total Events: {len(df)}")

# Events per month
df["year_month"] = df["event_date"].dt.to_period("M")
events_per_month = df.groupby("year_month").size()
fatalities_per_month = df.groupby("year_month")["fatalities"].sum()

# Visualize temporal patterns
fig, axes = plt.subplots(3, 1, figsize=(15, 10))

# Plot 1: Events over time
events_per_month.plot(ax=axes[0], title="Events per Month")
axes[0].set_ylabel("Number of Events")

# Plot 2: Fatalities over time
fatalities_per_month.plot(ax=axes[1], title="Fatalities per Month", color="red")
axes[1].set_ylabel("Total Fatalities")

# Plot 3: Distribution of fatalities
df.groupby("year_month")["fatalities"].mean().plot(
    ax=axes[2], title="Average Fatalities per Event", color="orange"
)
axes[2].set_ylabel("Avg Fatalities")

plt.tight_layout()
plt.savefig("temporal_patterns.png")
plt.show()

# Check for major phases/shifts
print("\nFatalities by Year:")
print(df.groupby(df["event_date"].dt.year)["fatalities"].agg(["sum", "mean", "count"]))

## Dataset Overview
- **Date Range:** January 1, 2022 - January 2, 2025
- **Total Duration:** 1,097 days (~3 years)
- **Total Events:** 158,099
- **Total Fatalities:** 151,380

## Key Observations

### 1. Clear War Phases

The conflict exhibits distinct temporal phases with varying intensity:

- **Early 2022 (Jan-Feb):** Rapid escalation period marking the invasion start
  - Sharp increase from near-zero to ~5,000 fatalities/month
  - Event frequency ramping up from ~500 to ~3,000 events/month

- **Mid 2022-2023:** Relatively stable operational period
  - Consistent ~3,000-4,000 fatalities/month
  - Stable event frequency of ~4,000-5,000 events/month
  - Average fatalities per event: 0.6-1.2

- **Mid 2024 onwards:** Major escalation phase
  - Dramatic increase to ~8,000+ fatalities/month (peak: ~9,000)
  - Sustained high-intensity period from May 2024 through December 2024
  - Average fatalities per event: 1.5-2.3

- **Jan 2025:** Apparent data drop-off
  - Only 275 events and 363 fatalities recorded
  - Likely incomplete data (only 2 days captured)
  - Should be excluded from analysis

### 2. Critical Pattern - 2024 Escalation

The most significant finding is the **regime change in mid-2024**:

- Fatalities approximately **doubled** starting May-June 2024
- This increase persisted through end of 2024 (7+ months)
- Represents a fundamental shift in conflict intensity
- **Implication:** Any predictive model must account for this distribution shift

**Statistical Evidence:**
- Pre-escalation (2022-2023): Mean = 0.74 fatalities/event
- Post-escalation (2024): Mean = 1.31 fatalities/event
- **~77% increase** in lethality per event

### 3. Events are Relatively Stable

Despite varying fatality counts, the **number of events remained consistent**:

- Steady state: ~4,000-5,500 events/month from mid-2022 through 2024
- This suggests:
  - Operational tempo remained constant
  - The escalation was driven by **deadlier engagements**, not more frequent ones
  - Geographic or tactical changes rather than increased activity

### 4. Average Fatalities per Event Increased

The **lethality per event** shows clear temporal evolution:

- **2022 Q1:** 0.2-1.7 fatalities/event (high variance during invasion)
- **2022 Q2 - 2023:** 0.6-1.2 fatalities/event (stabilized)
- **2024 Q1-Q2:** 0.7-1.0 fatalities/event (pre-escalation baseline)
- **2024 Q3-Q4:** 1.5-2.3 fatalities/event (escalated regime)

This pattern suggests:
- More intense combat operations in 2024
- Possible use of heavier weaponry or more concentrated attacks
- Shift from skirmishes to larger-scale engagements

## Implications for Modeling

### Distribution Shift Challenge
Models trained on 2022-2023 data will encounter significant **concept drift** when applied to 2024 data:
- Training distribution (2022-2023): μ = 0.74, relatively stable
- Test distribution (2024): μ = 1.31, new high-fatality regime
- Risk: Models may systematically **underpredict** in the new regime

### Recommended Approach
1. Use **temporal validation** (not random splits) to properly evaluate generalization
2. Consider **log transformation** to handle increasing variance
3. Monitor for **regime detection** in production systems
4. Plan for **periodic retraining** as conflict dynamics evolve

## Data Quality Note
**Exclude January 2025 data** from analysis:
- Only 2 days of data captured (275 events vs ~4,000+ typical monthly count)
- Would introduce artificial drop-off pattern
- Final valid date: December 31, 2024